# Atividade 1 — Aquisição e representação de imagens

## Modo de estudo

Este notebook deve ser executado célula por célula. Antes de cada execução:

1. escreva o que espera que aconteça;
2. execute somente a célula atual;
3. leia a saída e explique-a com suas palavras;
4. só então avance.

O notebook não executa células automaticamente e não contém conclusões prontas. A implementação completa anterior foi preservada separadamente em atividade_1_gabarito.ipynb.

### Objetivo geral

Investigar como resolução, espaço de cor, quantização e formato de arquivo alteram a representação de imagens das classes **contrabaixo** e **FPGA**.

## Mapa do problema

**Entradas:** cinco fotografias originais de cada classe.

**Experimentos:**

- resolução: 100%, 50% e 20%;
- espaços de cor: RGB, HSV e escala de cinza;
- quantização: 256, 64, 32 e 2 níveis;
- formatos: JPEG e PNG.

**Princípio experimental:** modificar apenas uma variável por vez. Assim, a causa das diferenças observadas permanece identificável.

**Fluxo de trabalho:** validar entradas → testar uma imagem → interpretar → generalizar para o dataset → registrar resultados.

## Checkpoint 1 — Preparação do ambiente

### Hipótese antes de executar

Escreva aqui quais bibliotecas você acredita que já estão instaladas e qual erro espera receber caso alguma esteja ausente.

**Minha hipótese:** 

In [1]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from PIL import Image, ImageOps

Se ocorrer ModuleNotFoundError, leia o nome do módulo ausente. Em um notebook Jupyter, a instalação pode ser feita em uma célula separada com:

    %pip install numpy pandas matplotlib pillow opencv-python

Depois da instalação, reinicie o kernel se o ambiente solicitar. No Google Colab, essas bibliotecas normalmente já estão disponíveis.

## Checkpoint 1.1 — Localizar a raiz do projeto

Selecione local se estiver no VS Code ou colab se estiver no Google Colab. No Colab, ajuste o caminho para corresponder exatamente ao nome das pastas no Google Drive; maiúsculas, minúsculas, espaços e sublinhados importam.

In [ ]:
AMBIENTE = "local"  # use "local" ou "colab"

if AMBIENTE == "colab":
    from google.colab import drive

    drive.mount("/content/drive")
    RAIZ = Path("/content/drive/MyDrive/projetos/VC_iREDE")
elif AMBIENTE == "local":
    RAIZ = Path.cwd()

    if RAIZ.name.lower() == "notebooks":
        RAIZ = RAIZ.parent
else:
    raise ValueError("AMBIENTE deve ser 'local' ou 'colab'.")

print("Diretório de execução:", Path.cwd())
print("Raiz calculada:", RAIZ)
print("A raiz existe?", RAIZ.exists())

### Perguntas de interpretação

1. Qual é a diferença entre o diretório de execução e a pasta onde o arquivo do notebook está salvo?
2. Por que usamos RAIZ.parent quando o diretório atual se chama notebooks?
3. Por que devemos interromper o experimento se RAIZ.exists() resultar em False?

**Minha resposta:** 

In [ ]:
if not RAIZ.exists():
    raise FileNotFoundError(
        f"O projeto não foi encontrado em: {RAIZ}"
    )

## Checkpoint 1.2 — Validar a estrutura de diretórios

O operador / de um objeto Path junta partes de um caminho. A célula seguinte somente constrói e verifica caminhos; ela ainda não processa imagens.

In [ ]:
PASTA_ORIGINAIS = RAIZ / "dataset" / "originais"

PASTAS_CLASSES = {
    "contrabaixo": PASTA_ORIGINAIS / "contrabaixo",
    "fpga": PASTA_ORIGINAIS / "fpga",
}

CAMINHO_METADATA = RAIZ / "metadata.csv"

for nome, caminho in PASTAS_CLASSES.items():
    print(f"{nome}: {caminho}")
    print(f"Existe? {caminho.exists()}")

print("metadata.csv existe?", CAMINHO_METADATA.exists())

## Checkpoint 1.3 — Listar somente imagens válidas

Antes de executar, responda:

- quantos arquivos devem ser encontrados em cada classe?
- em qual ordem devem aparecer?
- um arquivo de texto seria aceito pela função?

**Minha previsão:** 

In [ ]:
EXTENSOES_VALIDAS = {".jpg", ".jpeg", ".png"}


def listar_imagens(pasta):
    arquivos = []

    for caminho in pasta.iterdir():
        eh_arquivo = caminho.is_file()
        extensao = caminho.suffix.lower()

        if eh_arquivo and extensao in EXTENSOES_VALIDAS:
            arquivos.append(caminho)

    return sorted(arquivos)

In [ ]:
imagens_por_classe = {}

for classe, pasta in PASTAS_CLASSES.items():
    imagens = listar_imagens(pasta)
    imagens_por_classe[classe] = imagens

    print(f"\nClasse: {classe}")
    print(f"Quantidade: {len(imagens)}")

    for imagem in imagens:
        print(" -", imagem.name)

## Checkpoint 1.4 — Conferir quantidade e nomenclatura

Uma validação transforma uma suposição em uma condição verificável. Neste dataset, esperamos exatamente cinco arquivos corretamente nomeados em cada classe.

In [ ]:
nomes_esperados = {
    "contrabaixo": {
        f"contrabaixo_{numero:02d}.jpg"
        for numero in range(1, 6)
    },
    "fpga": {
        f"fpga_{numero:02d}.jpg"
        for numero in range(1, 6)
    },
}

for classe, imagens in imagens_por_classe.items():
    encontrados = {imagem.name for imagem in imagens}
    esperados = nomes_esperados[classe]

    faltantes = esperados - encontrados
    inesperados = encontrados - esperados

    print(f"\nClasse: {classe}")
    print("Faltantes:", faltantes)
    print("Inesperados:", inesperados)

    assert len(imagens) == 5, (
        f"A classe {classe} deveria ter 5 imagens, "
        f"mas possui {len(imagens)}."
    )
    assert encontrados == esperados, (
        f"A nomenclatura da classe {classe} não corresponde ao padrão."
    )

### Interpretação da validação

- esperados - encontrados produz os nomes que estão faltando.
- encontrados - esperados produz nomes extras ou incorretos.
- :02d formata os números como 01, 02, 03, 04 e 05.
- assert interrompe a execução quando uma condição obrigatória é falsa.

**O que observei:** 

## Checkpoint 1.5 — Inspecionar o arquivo de metadados

Primeiro visualizamos a tabela e seus nomes de colunas. Ainda não alteramos nem salvamos o CSV.

In [ ]:
metadata = pd.read_csv(CAMINHO_METADATA)

print("Dimensão da tabela:", metadata.shape)
print("Colunas:", metadata.columns.tolist())
print()
print(metadata.to_string(index=False))

### Perguntas sobre os metadados

1. O que cada linha representa?
2. O que cada coluna representa?
3. A quantidade de linhas corresponde à quantidade de imagens?
4. Há células vazias? Elas representam erro ou informação ainda desconhecida?

**Minha análise:** 

## Checkpoint 1.6 — Abrir uma imagem como matriz

A função abaixo abre uma imagem, aplica a orientação EXIF registrada pela câmera, garante o espaço RGB e converte o resultado em uma matriz NumPy.

In [ ]:
def carregar_rgb(caminho):
    with Image.open(caminho) as arquivo:
        imagem_corrigida = ImageOps.exif_transpose(arquivo)
        imagem_rgb = imagem_corrigida.convert("RGB")
        matriz = np.array(imagem_rgb)

    return matriz

Antes de executar a próxima célula, faça uma previsão:

- qual será a ordem das dimensões da matriz?
- quantos canais ela terá?
- qual deverá ser o tipo numérico?
- quais são os valores mínimo e máximo possíveis?

**Minha previsão:** 

In [ ]:
caminho_teste = imagens_por_classe["fpga"][0]
imagem = carregar_rgb(caminho_teste)

print("Arquivo:", caminho_teste.name)
print("Shape:", imagem.shape)
print("Tipo:", imagem.dtype)
print("Menor valor encontrado:", imagem.min())
print("Maior valor encontrado:", imagem.max())
print("Memória sem compressão:", imagem.nbytes / (1024 ** 2), "MiB")

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(imagem)
plt.title(caminho_teste.name)
plt.axis("off")
plt.show()

## Registro do Checkpoint 1

**Raiz encontrada:**

**Quantidade de imagens de contrabaixo:**

**Quantidade de imagens de FPGA:**

**Shape da imagem testada:**

**Tipo dos pixels:**

**Intervalo de valores observado:**

**Diferença entre minha previsão e o resultado:**

**Dúvidas:** 

---

## Checkpoint 2 — Resolução

Não iniciado. Antes de escrever o código, vamos deduzir como calcular as dimensões de 50% e 20%, escolher um método de interpolação e formular hipóteses sobre os detalhes que serão perdidos.

## Checkpoint 3 — Espaços de cor

Não iniciado. Vamos estudar RGB, HSV e escala de cinza antes das conversões.

## Checkpoint 4 — Quantização

Não iniciado. Vamos derivar matematicamente a redução para 256, 64, 32 e 2 níveis.

## Checkpoint 5 — Formatos JPEG e PNG

Não iniciado. Vamos controlar as condições do experimento e comparar tamanho, perda e artefatos.

## Checkpoint 6 — Relato dos resultados

Não iniciado. As conclusões serão escritas a partir das observações registradas nos checkpoints anteriores.

## Fontes para consulta

- Python pathlib: https://docs.python.org/3/library/pathlib.html
- Pandas read_csv: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- Pillow ImageOps.exif_transpose: https://pillow.readthedocs.io/en/stable/reference/ImageOps.html#PIL.ImageOps.exif_transpose
- OpenCV — transformações geométricas: https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html
- OpenCV — conversões de cor: https://docs.opencv.org/4.x/d8/d01/group__imgproc__color__conversions.html